In [1]:
import pandas as pd
import numpy as np
import os
import math
import matplotlib.pyplot as plt
from DaySim_Survey import DaysimSummary

In [2]:
filepath ="C:/Users/USVA682771/OneDrive - WSP O365/Documents/client_chattanooga_TNHTS/data/processed"
perdata = pd.read_csv(os.path.join(filepath, "a_survey_per.csv"), sep = ',')
hhdata = pd.read_csv(os.path.join(filepath, "a_survey_hh.csv"), sep = ',')
tripdata = pd.read_csv(os.path.join(filepath, "a_survey_place.csv"), sep = ',')
tourdata = pd.read_csv(os.path.join(filepath, "a_survey_tour.csv"), sep = ',')


C:\Users\USVA682771\AppData\Local\Temp\ipykernel_11992\2814327064.py:2: DtypeWarning: Columns (0: occup2_o) have mixed types. Specify dtype option on import or set low_memory=False.
  perdata = pd.read_csv(os.path.join(filepath, "a_survey_per.csv"), sep = ',')


In [3]:
hhdata.columns

Index(['sampno', 'travel_date', 'travday', 'hhsize', 'hhveh', 'homeown',
       'homeown_o', 'resty', 'resty_o', 'hhinc', 'hhinc_full', 'trvl_ex_walk',
       'trvl_ex_bike', 'trvl_ex_bikeshare', 'trvl_ex_escooter', 'trvl_ex_taxi',
       'trvl_ex_rideshare', 'trvl_ex_transit', 'techimpct_telework',
       'techimpct_e_school', 'techimpct_shoponline', 'techimpct_drv4wrk',
       'techimpct_telehealth', 'numbicycle', 'ebikes', 'hghspdint', 'acclvl2',
       'td_spring_break', 'tripsub', 'incen_choice', 'futuresurvey', 'mpo',
       'hhresp', 'hhsize_completed', 'hhtrips'],
      dtype='str')

In [4]:
perdata.columns

Index(['sampno', 'perno', 'age', 'aage', 'age16', 'age75', 'relate',
       'relate_o', 'gender', 'gender_o',
       ...
       'nogowhy_dk', 'nogowhy_o', 'proxy', 'whoproxy', 'wkstat2',
       'partial_complete', 'qc_trip_person', 'pertrips', 'app_installed',
       'app_trips_reported'],
      dtype='str', length=306)

In [5]:
tripdata.columns

Index(['sampno', 'perno', 'traveldayno', 'placeno', 'locno', 'arrtime',
       'deptime', 'travtime', 'actdur', 'distance', 'mode', 'mode_o', 'vehno',
       'tpurp', 'tpurp_o', 'cravl', 'fare', 'payf', 'payf_o', 'paypk', 'pkamt',
       'pkbas', 'pkbas_o', 'prkty', 'prkty_o', 'prkty_loc', 'tnc_fare',
       'party', 'hhparty', 'hhmem_count', 'nonhhmem_count', 'companions',
       'loop_trip', 'source_mode', 'qc_loc', 'qc_joint', 'qc_time', 'qc_loop',
       'qc_mode', 'qc_day', 'perno_1', 'perno_2', 'perno_3', 'perno_4',
       'perno_5', 'perno_6', 'perno_7', 'perno_8', 'perno_9', 'perno_10',
       'perno_11', 'perno_12', 'placeid', 'trpwts2g0', 'tourid', 'ltripid'],
      dtype='str')

In [6]:
tourdata.columns

Index(['sampno', 'perno', 'tourid', 'primtrip1', 'primtrip2', 'primtrip3',
       'primtrip4', 'primloc', 'lastprimtrip', 'primpurp', 'stops_o',
       'stops_i', 'escort_o', 'escort_i', 'numsubtours', 'o_dep', 'o_arr',
       'i_dep', 'i_arr', 'homestart', 'homeend', 'modeout', 'modein',
       'outmodepriority', 'inmodepriority', 'tourmode', 'validity_flag',
       'tourwts2g0'],
      dtype='str')

In [7]:
# Map survey column names to DaySim expected column names
perdata_mapped = perdata.copy()
hhdata_mapped = hhdata.copy()
tripdata_mapped = tripdata.copy()
tourdata_mapped = tourdata.copy()

# Prefer harmonized worker-status field if available
if 'wkstat2' in perdata_mapped.columns:
    perdata_mapped['wkstat'] = perdata_mapped['wkstat2']

# Add DaySim-like aliases that are commonly used in DaySim_Survey.py
if 'resty' in hhdata_mapped.columns and 'hrestype' not in hhdata_mapped.columns:
    hhdata_mapped['hrestype'] = hhdata_mapped['resty']

# hhtaz is required by several summaries; fallback to mpo if no TAZ field exists
if 'hhtaz' not in hhdata_mapped.columns:
    hhdata_mapped['hhtaz'] = hhdata_mapped['mpo'] if 'mpo' in hhdata_mapped.columns else np.nan

# Ensure expected source columns exist for downstream rename step
for col in ['sampno', 'perno', 'age', 'wkstat']:
    if col not in perdata_mapped.columns:
        perdata_mapped[col] = np.nan

for col in ['sampno', 'hhinc']:
    if col not in hhdata_mapped.columns:
        hhdata_mapped[col] = np.nan

# Rename person columns to match DaySim format
perdata_mapped = perdata_mapped.rename(columns={
    'sampno': 'hhno',
    'perno': 'pno',
    'wkstat': 'pptyp',  # adjust if different column maps to pptyp
    'age': 'pagey',
})

# Add psexpfac if not present (use uniform weight of 1.0 as placeholder)
if 'psexpfac' not in perdata_mapped.columns:
    perdata_mapped['psexpfac'] = 1.0

# Rename household columns to match DaySim format
hhdata_mapped = hhdata_mapped.rename(columns={
    'sampno': 'hhno',
    'hhinc': 'hhincome',
})

# Add hhexpfac if not present
if 'hhexpfac' not in hhdata_mapped.columns:
    hhdata_mapped['hhexpfac'] = 1.0

report = DaysimSummary(perdata=perdata_mapped, hhdata=hhdata_mapped, tripdata=tripdata_mapped, tourdata=tourdata_mapped)

TypeError: DaysimSummary.__init__() got an unexpected keyword argument 'perdata'

## Vehicle Availability

In [ ]:
# by household income
report.summary_vehavail("inccat")

In [ ]:
# by county
report.summary_vehavail("hhcounty")

In [ ]:
# by household drivers
report.summary_vehavail("hh16cat")

In [ ]:
# Vehicle ownership in the report (exclude GQ)
hhdata = report.hhdata
perdata = report.perdata

hhdata = hhdata[hhdata["hrestype"]!=9].copy()
hhdata["hhvehcat"] = np.where(hhdata.hhveh>4, 4, hhdata.hhveh)
perdata["hh16cat"] = np.where(perdata.pagey>=16, 1, 0)  #potential drivers
aggper = perdata.groupby("hhno")["hh16cat"].sum() 
hhdata = pd.merge(hhdata, aggper, on="hhno", how="left")
hhdata["hh16cat"] = np.where(hhdata.hh16cat>4, 4, hhdata.hh16cat)
hhdata["inccat"] = pd.cut(hhdata["hhincome"], 
                      bins=[0,25000,50000,75000,float("inf")], 
                      labels=["0K-25K", "25K-50K", "50K-75K", ">75K"], 
                      right=False)
hhdata = (hhdata.merge(report.countycorr, 
                      left_on="hhtaz", 
                      right_on="TAZID", 
                      how="left").
                      rename(columns={"District": "hhcounty"}))

In [ ]:
# hhdata[hhdata["hrestype"]==9]

(hhdata.groupby(["hh16cat","hhvehcat"])["hhexpfac"].
                            sum().
                            reset_index().
                            pivot_table(values="hhexpfac", 
                                        index="hh16cat",
                                        columns="hhvehcat",
                                        fill_value=0))

In [ ]:

(hhdata.groupby(["hhwkrs","hhvehcat"])["hhexpfac"].
                            sum().
                            reset_index().
                            pivot_table(values="hhexpfac", 
                                        index="hhwkrs",
                                        columns="hhvehcat",
                                        fill_value=0))

In [ ]:

(hhdata.groupby(["inccat","hhvehcat"])["hhexpfac"].
                            sum().
                            reset_index().
                            pivot_table(values="hhexpfac", 
                                        index="inccat",
                                        columns="hhvehcat",
                                        fill_value=0))

## Work/School Location

In [ ]:
# work trip length
df = report.summary_wrkschloc_trip_length("wrkr")
df.columns = df.columns.astype(str)
df = df.reset_index()
# plt.bar(x=df['wrkdistcat'],height=(df['FT']+df['PT']+df['NotFTPT']))

In [ ]:
# away from home full-time workers have a work location that is the same TAZ as their home location
perdata = report.perdata_wrkschloc
perdata = perdata[perdata['wfh']==0]
perdata = perdata[(perdata.wrkr==1) & (perdata.hhtaz==perdata.pwtaz)]

report.summary_func(perdata, "wrkdistcat", "wrkrtyp", "psexpfac", subsetvar='wrkr', subsetval=1).to_clipboard()

In [ ]:
# exclude work from home trips
perdata = report.perdata_wrkschloc
perdata = perdata[perdata['sfh']==0]
report.summary_func(perdata, "schdistcat", "stutyp", "psexpfac", subsetvar='stud', subsetval=1)

In [ ]:
# work trip duration
df = report.summary_wrkschloc_trip_duration("wrkr")
df.columns = df.columns.astype(str)
df = df.reset_index()
plt.bar(x=df['wrktimecat'],height=(df['FT']+df['PT']+df['NotFTPT']))

In [ ]:
# work trip county flow
report.summary_wrkschloc_county_flow("wrkr")

In [ ]:
# work from home
report.summary_wrkschloc_at_home("wfh")

In [ ]:
# school trip length
df = report.summary_wrkschloc_trip_length("stud")
df.columns = df.columns.astype(str)
df = df.reset_index().to_clipboard()
# plt.bar(x=df['schdistcat'],height=(df['Ch515']+df['Stu16']+df['UniStu']))

In [ ]:
# exclude work from home trips
perdata = report.perdata_wrkschloc
perdata = perdata[perdata['sfh']==0]
report.summary_func(perdata, "schdistcat", "stutyp", "psexpfac", subsetvar='stud', subsetval=1).to_clipboard()

In [ ]:
# exclude school from home trips
# school in GA and TN
countycorr = pd.read_csv(r'C:\Users\USYS671257\OneDrive - WSP O365\21_31000110.002_Chattanooga TPO Model\2-DaySim_summary_report'+"/county_districts_chattanooga.csv").rename(columns={'STATEID':"District"})
countycorr_dict = countycorr.set_index("TAZID")["District"].to_dict()

perdata = report.perdata_wrkschloc.copy()
# perdata["pscounty"] = perdata["pstaz"].map(countycorr_dict)
perdata["pscounty"] = perdata["hhtaz"].map(countycorr_dict)
perdata = perdata[perdata['sfh']==0]
report.summary_func(perdata, "schdistcat", "pscounty", "psexpfac", subsetvar='stud', subsetval=1).to_clipboard()

In [ ]:
# school trip duration
df = report.summary_wrkschloc_trip_duration("stud")
df.columns = df.columns.astype(str)
df = df.reset_index()
# plt.bar(x=df['schtimecat'],height=(df['Ch515']+df['Stu16']+df['UniStu']))

In [ ]:
# school trip county flow
report.summary_wrkschloc_county_flow("stud")

In [ ]:
# school at home
report.summary_wrkschloc_at_home("sfh")

## Trip Mode

In [ ]:
# Work
report.summary_trip_mode(purpose=1) 
# 1-Work 2-School 3-Escort 4-Personal_Business 5-Shop 6-Meal 7-Social&Recreational 8-Workbased

## Tour Mode

In [ ]:
# Work
report.summary_tour_mode(purpose=8)
# 1-Work 2-School 3-Escort 4-Personal_Business 5-Shop 6-Meal 7-Social&Recreational 8-Workbased

## Trip Destination

In [ ]:
# trip length by trip purpose
df = report.summary_trip_destination("distcat").reset_index()
plt.bar(x=df['distcat'],height=df[3]) 
# 0-non/home  1-Work 2-School 3-Escort 4-Personal_Business 5-Shop 6-Meal 7-Social&Recreational 10-change mode inserted purpose

In [ ]:
tripdata = pd.read_csv(r"C:\Users\USYS671257\OneDrive - WSP O365\21_31000110.002_Chattanooga TPO Model\model outputs\survey" + "/chc_trip_new.dat", sep = ' ')
a = tripdata.copy()
a = a[(a["travdist"]!=0)]
a["distcat"] = pd.cut(a["travdist"], 
                                bins=range(0, 91),  
                                right=False,
                                labels=list(range(1, 91)))

In [ ]:
(a.groupby(["distcat","dpurp"])["trexpfac"].
                    sum().
                    reset_index().
                    pivot_table(values="trexpfac", 
                                index="distcat",
                                columns="dpurp",
                                fill_value=0))

In [ ]:
# trip duration by trip purpose
df = report.summary_trip_destination("timecat").reset_index()
plt.bar(x=df['timecat'],height=df[0])
# 0-non/home  1-Work 2-School 3-Escort 4-Personal_Business 5-Shop 6-Meal 7-Social&Recreational 10-change mode inserted purpose

## Tour Destination

In [ ]:
# tour length by tour purpose
df = report.summary_tour_destination("distcat").reset_index().to_clipboard()
# plt.bar(x=df['distcat'],height=df[1])
# 1-Work 2-School 3-Escort 4-Personal_Business 5-Shop 6-Meal 7-Social&Recreational 8-Workbased

In [ ]:
tourdata = pd.read_csv(r"C:\Users\USYS671257\OneDrive - WSP O365\21_31000110.002_Chattanooga TPO Model\model outputs\survey" + "/chc_tour_new.dat", sep = ' ')
a = tourdata[["totaz","tdtaz","pdpurp","parent","tautodist","tautocost","tautotime","toexpfac"]]
a = a[(a["tautodist"]!=0)]
a["pdpurp"] = np.where(a.pdpurp==8, 7, a.pdpurp)   # combine recreational 8 with socail 7
a["pdpurp"] = np.where(a.pdpurp==9, 4, a.pdpurp)   # combine medical 8 with personal business 4
a["pdpurp2"] = np.where(a.parent==0, a.pdpurp, 8)   # workbased trips
a["distcat"] = pd.cut(a["tautodist"], 
                                bins=range(0, 91),  
                                right=False,
                                labels=list(range(1, 91)))

In [ ]:
(a.groupby(["distcat","pdpurp"])["toexpfac"].
                    sum().
                    reset_index().
                    pivot_table(values="toexpfac", 
                                index="distcat",
                                columns="pdpurp",
                                fill_value=0))

In [ ]:
# Section 5.1 Tour Frequecy
countycorr_dict = report.countycorr.set_index("TAZID")["District"].to_dict()
hhdata = report.hhdata
perdata = report.perdata
tourdata = report.tourdata

hhdata["hhcounty"] = hhdata["hhtaz"].map(countycorr_dict)
perdata = perdata.merge(hhdata, on="hhno", how="left")
perdata = perdata[["hhno","pno","pptyp","hhtaz","hhcounty","pwtaz","psexpfac"]]

tourdata = pd.merge(tourdata, perdata, on=["hhno","pno"], how="left")
if report.excludeChildren5:
    tourdata = tourdata[tourdata["pptyp"]<8]

tourdata["pdpurp"] = np.where(tourdata.pdpurp==8, 7, tourdata.pdpurp)   # combine recreational 8 with socail 7
tourdata["pdpurp"] = np.where(tourdata.pdpurp==9, 4, tourdata.pdpurp)   # combine medical 8 with personal business 4
tourdata["pdpurp2"] = np.where(tourdata.parent==0, tourdata.pdpurp, 8)   # workbased trips
tourdata["ocounty"] = tourdata["totaz"].map(countycorr_dict)
tourdata["dcounty"] = tourdata["tdtaz"].map(countycorr_dict)
tourdata["distcat"] = pd.cut(tourdata["tautodist"], 
                                bins=range(0, 91),  
                                right=False,
                                labels=list(range(1, 91)))
tourdata["timecat"] = pd.cut(tourdata["tautotime"], 
                                bins=range(0, 91),  
                                right=False,
                                labels=list(range(1, 91)))
tourdata["wrkrtyp"] = np.where(tourdata.pptyp==1, "FT", 
                               np.where(tourdata.pptyp==2, "PT","NotFTPT"))
tourdata["wrkrtyp"] = tourdata["wrkrtyp"].astype(pd.CategoricalDtype(categories=["FT","PT","NotFTPT"]))
tourdata["tautodist"] = np.where(tourdata.tautodist<0, np.NaN, tourdata.tautodist)
tourdata["tautotime"] = np.where(tourdata.tautotime<0, np.NaN, tourdata.tautotime)

In [ ]:
# Mandantory
# tourfreq = tourdata[(tourdata["pdpurp2"]<=2)].copy()
tourfreq = tourdata.copy()
tourfreq["Work"] = np.where(tourfreq.pdpurp2==1, 1, 0)
tourfreq["School"] = np.where(tourfreq.pdpurp2==2, 1, 0)
tourfreq = tourfreq.groupby(["hhno","pno","pptyp"])["Work","School"].sum().reset_index()

tourfreq["Type"] = None
tourfreq.loc[((tourfreq.Work==0) & (tourfreq.School==0)),"Type"] = "0 Work Tour + 0 School Tour"
tourfreq.loc[((tourfreq.Work==0) & (tourfreq.School==1)),"Type"] = "0 Work Tour + 1 School Tour"
tourfreq.loc[((tourfreq.Work==0) & (tourfreq.School>=2)),"Type"] = "0 Work Tour + 2+ School Tour"
tourfreq.loc[((tourfreq.Work==1) & (tourfreq.School==0)),"Type"] = "1 Work Tour + 0 School Tour"
tourfreq.loc[((tourfreq.Work==1) & (tourfreq.School==1)),"Type"] = "1 Work Tour + 1 School Tour"
tourfreq.loc[((tourfreq.Work==1) & (tourfreq.School>=2)),"Type"] = "1 Work Tour + 2+ School Tour"
tourfreq.loc[((tourfreq.Work>=2) & (tourfreq.School==0)),"Type"] = "2+ Work Tour + 0 School Tour"
tourfreq.loc[((tourfreq.Work>=2) & (tourfreq.School==1)),"Type"] = "2+ Work Tour + 1 School Tour"
tourfreq.loc[((tourfreq.Work>=2) & (tourfreq.School>=2)),"Type"] = "2+ Work Tour + 2+ School Tour"

tourfreq =  tourfreq.merge(perdata[["hhno","pno","psexpfac"]], on=["hhno","pno"], how="left")

tourfreq = tourfreq.groupby(["pptyp","Type"])["psexpfac"].sum().reset_index()
tourfreq.pivot_table(values="psexpfac", index="pptyp", columns="Type", aggfunc=["mean"], fill_value=0)

In [ ]:
# Mandantory -- another way to calculate freq (work-based trips will be included)
hhdata = report.hhdata
perdata = report.perdata

hhdata["hhcounty"] = hhdata["hhtaz"].map(countycorr_dict)
perdata = perdata.merge(hhdata, on="hhno", how="left")
perdata = perdata[["hhno","pno","pptyp","hhtaz","hhcounty","pwtaz","psexpfac"]]

tourfreq = report.pdaydata.copy()
tourfreq = pd.merge(tourfreq, perdata, on=["hhno","pno"], how="left")
if report.excludeChildren5:
    tourfreq = tourfreq[tourfreq["pptyp"]<8]
    
tourfreq = tourfreq.groupby(["hhno","pno","pptyp"])["wktours","sctours"].sum().reset_index()

tourfreq["Type"] = None
tourfreq.loc[((tourfreq.wktours==0) & (tourfreq.sctours==0)),"Type"] = "0 Work Tour + 0 School Tour"
tourfreq.loc[((tourfreq.wktours==0) & (tourfreq.sctours==1)),"Type"] = "0 Work Tour + 1 School Tour"
tourfreq.loc[((tourfreq.wktours==0) & (tourfreq.sctours>=2)),"Type"] = "0 Work Tour + 2+ School Tour"
tourfreq.loc[((tourfreq.wktours==1) & (tourfreq.sctours==0)),"Type"] = "1 Work Tour + 0 School Tour"
tourfreq.loc[((tourfreq.wktours==1) & (tourfreq.sctours==1)),"Type"] = "1 Work Tour + 1 School Tour"
tourfreq.loc[((tourfreq.wktours==1) & (tourfreq.sctours>=2)),"Type"] = "1 Work Tour + 2+ School Tour"
tourfreq.loc[((tourfreq.wktours>=2) & (tourfreq.sctours==0)),"Type"] = "2+ Work Tour + 0 School Tour"
tourfreq.loc[((tourfreq.wktours>=2) & (tourfreq.sctours==1)),"Type"] = "2+ Work Tour + 1 School Tour"
tourfreq.loc[((tourfreq.wktours>=2) & (tourfreq.sctours>=2)),"Type"] = "2+ Work Tour + 2+ School Tour"

# tourfreq =  tourfreq.merge(perdata[["hhno","pno","psexpfac"]], on=["hhno","pno"], how="left")

# tourfreq = tourfreq.groupby(["pptyp","Type"])["psexpfac"].sum().reset_index()
# tourfreq.pivot_table(values="psexpfac", index="pptyp", columns="Type", aggfunc=["mean"], fill_value=0)

In [ ]:
# tour duration by tour purpose
df = report.summary_tour_destination("timecat").reset_index()
plt.bar(x=df['timecat'],height=df[1])
# 1-Work 2-School 3-Escort 4-Personal_Business 5-Shop 6-Meal 7-Social&Recreational 8-Workbased

In [ ]:
# tour county flow by tour purpose
report.summary_tour_destination_county_flow(purpose=7)
# 1-Work 2-School 3-Escort 4-Personal_Business 5-Shop 6-Meal 7-Social&Recreational 8-Workbased

In [ ]:
# tour purpose
tourpurp = report.tourdata_tour_destination.copy()
tourpurp["Type"] = None
tourpurp.loc[(tourpurp.pptyp<=2),"Type"] = "Workers"
tourpurp.loc[((tourpurp.pptyp==3) | (tourpurp.pptyp==4)),"Type"] = "Non working Adult"
tourpurp.loc[(tourpurp.pptyp==5),"Type"] = "University"
tourpurp.loc[(tourpurp.pptyp==6),"Type"] = "High school student age 16+"
tourpurp.loc[(tourpurp.pptyp>=7),"Type"] = "Child age below 16"
tourpurp = tourpurp.groupby(['Type',"pdpurp2"])["psexpfac"].sum().reset_index()
tourpurp['percent'] = tourpurp.groupby('Type')['psexpfac'].transform(lambda x: x/x.sum())

In [ ]:
tourpurp.to_clipboard()

In [ ]:
tourpurp = report.tourdata_tour_destination.copy()
tourpurp["Type"] = None
tourpurp.loc[(tourpurp.pptyp<=2),"Type"] = "Workers"
tourpurp.loc[((tourpurp.pptyp==3) | (tourpurp.pptyp==4)),"Type"] = "Non working Adult"
tourpurp.loc[(tourpurp.pptyp==5),"Type"] = "University"
tourpurp.loc[(tourpurp.pptyp==6),"Type"] = "High school student age 16+"
tourpurp.loc[(tourpurp.pptyp>=7),"Type"] = "Child age below 16"
tourpurp.groupby(["pdpurp2"])["psexpfac"].sum().reset_index()

## Trip Time of Day

In [ ]:
# Trip arrival time at stop location - outbound (first) half-tour
df = report.summary_trip_tod("arrtimecat",filter_by_var="arrflag").reset_index()
plt.bar(x=df['arrtimecat'],height=df[1])
# 1-Work 2-School 3-Escort 4-Personal_Business 5-Shop 6-Meal 7-Social&Recreational 10-change mode inserted purpose

In [ ]:
# Trip departure time at stop location - return (second) half-tour
df = report.summary_trip_tod("deptimecat",filter_by_var="depflag").reset_index()
plt.bar(x=df['deptimecat'],height=df[1])
# 1-Work 2-School 3-Escort 4-Personal_Business 5-Shop 6-Meal 7-Social&Recreational 10-change mode inserted purpose

In [ ]:
# Duration at stop location - both half-tours
df = report.summary_trip_tod("durdestcat",filter_by_var="durflag").reset_index()
plt.bar(x=df['durdestcat'],height=df[1])
# 1-Work 2-School 3-Escort 4-Personal_Business 5-Shop 6-Meal 7-Social&Recreational 10-change mode inserted purpose

In [ ]:
# Trip arrival time All
df = report.summary_trip_tod("arrtimecat",filter_by_var=False).reset_index()
plt.bar(x=df['arrtimecat'],height=df[1])
# 1-Work 2-School 3-Escort 4-Personal_Business 5-Shop 6-Meal 7-Social&Recreational 10-change mode inserted purpose

In [ ]:
# Trip departure time All
df = report.summary_trip_tod("deptimecat",filter_by_var=False).reset_index()
plt.bar(x=df['deptimecat'],height=df[1])
# 1-Work 2-School 3-Escort 4-Personal_Business 5-Shop 6-Meal 7-Social&Recreational 10-change mode inserted purpose

## Tour Time of Day

In [ ]:
# arrival time all trip purposes
df = report.summary_tour_tod("arrtimecat").reset_index()
plt.bar(x=df['arrtimecat'],height=df[1]) 
# 1 Work 2 School 3 Other 4 Workbased

In [ ]:
a = report.tourdata_tour_tod

In [ ]:
a[["tlvdest","deppdhr","deppdmin","tardest","arrpdhr","arrpdmin","deptime","arrtime","arrtimecat","pdpurp2"]]

In [ ]:
a[(a["arrtimecat"]==3) & (a["pdpurp2"]==1)]["psexpfac"]

In [ ]:
df

In [ ]:
# departure time all trip purposes
df = report.summary_tour_tod("deptimecat").reset_index()
plt.bar(x=df['deptimecat'],height=df[1]) 
# 1 Work 2 School 3 Other 4 Workbased

In [ ]:
# duration all trip purposes
df = report.summary_tour_tod("durdestcat").reset_index().to_clipboard()
# plt.bar(x=df['durdestcat'],height=df[2]) 
# 1 Work 2 School 3 Other 4 Workbased

In [ ]:
# arrival time by trip purpose by person type
report.summary_tour_tod_purpose("arrtimecat", purpose=2).reset_index()
# purpose: 1 Work 2 School 3 Other 4 Workbased
# columns: 1-FT 2-PT 3-Retired 4-Nonworker 5-UnivStud 6-Stud16+ 7-Stud5-15 8-Child<5

In [ ]:
# departure time by trip purpose by person type
report.summary_tour_tod_purpose("deptimecat", purpose=2).reset_index()
# purpose: 1 Work 2 School 3 Other 4 Workbased
# columns: 1-FT 2-PT 3-Retired 4-Nonworker 5-UnivStud 6-Stud16+ 7-Stud5-15 8-Child<5

## Person Day Pattern

In [ ]:
# number of tours by person type
report.summary_day_pattern_num_of_tours()
# columns: 1-FT 2-PT 3-Retired 4-Nonworker 5-UnivStud 6-Stud16+ 7-Stud5-15 8-Child<5

In [ ]:
# tour/stop combinations by person type
report.summary_day_pattern_tour_stops()
# columns: 1-FT 2-PT 3-Retired 4-Nonworker 5-UnivStud 6-Stud16+ 7-Stud5-15 8-Child<5
# index: tours/stops
# 0 0/0
# 1 1/0
# 2 1/1
# 3 1/2
# 4 1/3+
# 5 2/0
# 6 2/1
# 7 2/2
# 8 2/3+
# 9 3+/0
# 10 3+/1
# 11 3+/2
# 12 3+/3+

In [ ]:
# tour/stop combinations by purpose by person type
report.summary_day_pattern_tour_stops_by_purpose(purpose="wktostp")
# purpose: "wktostp", "sctostp", "estostp", "pbtostp", "shtostp", "mlstops", "sotostp"
# columns: 1-FT 2-PT 3-Retired 4-Nonworker 5-UnivStud 6-Stud16+ 7-Stud5-15 8-Child<5
# index: tours/stops
# 1 0/0
# 2 0/1+
# 3 1+/0
# 4 1+/1+

In [ ]:
# tours by purpose by person type
report.summary_day_pattern_tours_by_purpose(purpose="wktopt")
# purpose: "wktopt", "sctopt", "estopt", "pbtopt", "shtopt", "mltopt", "sotopt"
# columns: 1-FT 2-PT 3-Retired 4-Nonworker 5-UnivStud 6-Stud16+ 7-Stud5-15 8-Child<5
# index: 0,1,2,3+

In [ ]:
# number of subtours (work tours only)
report.summary_day_pattern_subtours()
# columns: 1-FT 2-Other
# index: 0,1,2,3+

In [ ]:
# number of subtours (work tours only) by purpose
report.summary_day_pattern_subtours_by_purpose()
# columns: 1-FT 2-Other
# index :1-Work 2-School 3-Escort 4-Personal_Business 5-Shop 6-Meal 7-Social&Recreational

In [ ]:
# estimated tours by number of stops and purpose
report.summary_day_pattern_stops_by_tour_purpose("stopscat")
# columns: 1-Work 2-School 3-Escort 4-Personal_Business 5-Shop 6-Meal 7-Social&Recreational
# index: 0,1,2,3,4,5,6+

In [ ]:
# estimated outbound tours by number of stops and purpose
report.summary_day_pattern_stops_by_tour_purpose("h1stopscat")
# columns: 1-Work 2-School 3-Escort 4-Personal_Business 5-Shop 6-Meal 7-Social&Recreational
# index: 0,1,2,3,4,5,6+

In [ ]:
# estimated return tours by number of stops and purpose
report.summary_day_pattern_stops_by_tour_purpose("h2stopscat")
# columns: 1-Work 2-School 3-Escort 4-Personal_Business 5-Shop 6-Meal 7-Social&Recreational
# index: 0,1,2,3,4,5,6+

In [ ]:
# estimated total tours by purpose by person type
report.summary_day_pattern_tours_by_tour_purpose("pptyp")
# columns: 1-FT 2-PT 3-Retired 4-Nonworker 5-UnivStud 6-Stud16+ 7-Stud5-15 8-Child<5
# index: 1-Work 2-School 3-Escort 4-Personal_Business 5-Shop 6-Meal 7-Social&Recreational 8-Workbased

In [ ]:
# estimated total tours by purpose by household income
report.summary_day_pattern_tours_by_tour_purpose("inccat")
# index: 1-Work 2-School 3-Escort 4-Personal_Business 5-Shop 6-Meal 7-Social&Recreational 8-Workbased

In [ ]:
# estimated total tours by purpose by auto sufficiency
report.summary_day_pattern_tours_by_tour_purpose("vehsuf")
# columns: 1-0, 2-autos<drivers, 3-autos=drivers, 4-autos>drivers
# index: 1-Work 2-School 3-Escort 4-Personal_Business 5-Shop 6-Meal 7-Social&Recreational 8-Workbased

In [ ]:
# estimated total tours by purpose by county
report.summary_day_pattern_tours_by_tour_purpose("hhcounty")
# index: 1-Work 2-School 3-Escort 4-Personal_Business 5-Shop 6-Meal 7-Social&Recreational 8-Workbased

In [ ]:
# estimated total stops by purpose by person type
report.summary_day_pattern_stops_by_stop_purpose_agg("pptyp")
# columns: 1-FT 2-PT 3-Retired 4-Nonworker 5-UnivStud 6-Stud16+ 7-Stud5-15 8-Child<5

In [ ]:
# estimated total stops by purpose by household income
report.summary_day_pattern_stops_by_stop_purpose_agg("inccat")

In [ ]:
# estimated total stops by purpose by auto sufficiency
report.summary_day_pattern_stops_by_stop_purpose_agg("vehsuf")
# columns: 1-0, 2-autos<drivers, 3-autos=drivers, 4-autos>drivers

In [ ]:
# estimated total stops by purpose by county
report.summary_day_pattern_stops_by_stop_purpose_agg("hhcounty")

In [ ]:
# estimated total trips by purpose by person type
report.summary_day_pattern_trips_by_destination_purpose("pptyp")
# columns: 1-FT 2-PT 3-Retired 4-Nonworker 5-UnivStud 6-Stud16+ 7-Stud5-15 8-Child<5

In [ ]:
# estimated total trips by purpose by household income
report.summary_day_pattern_trips_by_destination_purpose("inccat")

In [ ]:
# estimated total trips by purpose by auto sufficiency
report.summary_day_pattern_trips_by_destination_purpose("vehsuf")
# columns: 1-0, 2-autos<drivers, 3-autos=drivers, 4-autos>drivers

In [ ]:
# estimated total trips by purpose by county
report.summary_day_pattern_trips_by_destination_purpose("ocounty")

## Tour Trip Rates

In [ ]:
pday = report.pdaydata_day_pattern_pday
ptour = report.pdaydata_day_pattern_tour
ptrip = report.pdaydata_day_pattern_trip
hh = report.hhdata
per = report.perdata

In [ ]:
# hh tour rates
avgtour = ptour.groupby("hhno")["tour"].size().reset_index()
avgtour = avgtour.merge(hh[["hhno","hhexpfac"]], on="hhno", how="left")
np.average(avgtour['tour'], weights=avgtour['hhexpfac'])

In [ ]:
# hh trip rates
avgtrip = ptrip.groupby("hhno")["tour"].size().reset_index()
avgtrip = avgtrip.merge(hh[["hhno","hhexpfac"]], on="hhno", how="left")
np.average(avgtrip['tour'], weights=avgtrip['hhexpfac'])

In [ ]:
# person trip rates
avgtrip = ptrip.groupby(["hhno","pno"])["tour"].size().reset_index()
avgtrip = avgtrip.merge(per[["hhno","pno","psexpfac"]], on=["hhno","pno"], how="left")
np.average(avgtrip['tour'], weights=avgtrip['psexpfac'])

## Trip Generation Rates

In [ ]:
ptrip = report.pdaydata_day_pattern_trip
hh = report.hhdata
per = report.perdata

In [ ]:
ptrip = ptrip.merge(hh[["hhno","hhtaz"]], on="hhno", how="left")

In [ ]:
ptrip["TripType"] = "NHB"

ptrip.loc[((ptrip["otaz"]==ptrip["hhtaz"]) | (ptrip["dtaz"]==ptrip["hhtaz"])) & 
          ((ptrip["dpurp"]==1) | (ptrip["opurp"]==1)),
          "TripType"
         ] = "HBW"

ptrip.loc[((ptrip["otaz"]==ptrip["hhtaz"]) | (ptrip["dtaz"]==ptrip["hhtaz"])) & 
          ((ptrip["dpurp"]!=1) & (ptrip["opurp"]!=1)),
          "TripType"
         ] = "HBO"


In [ ]:
ptrip = ptrip[(ptrip["TripType"]=="HBW") & (ptrip["pptyp"]<=2)]

In [ ]:
avgtrip = ptrip.groupby(["hhno","pno"])["TripType"].size().reset_index()
avgtrip = avgtrip.merge(per[["hhno","pno","psexpfac"]], on=["hhno","pno"], how="left")
np.average(avgtrip['TripType'], weights=avgtrip['psexpfac'])

In [ ]:
avgtrip